In [ ]:
import os
print(os.getcwd())
 

In [ ]:
%%writefile app.py
from flask import Flask, jsonify
import os
import datetime

app = Flask(__name__)

@app.route('/')
def home():
    return jsonify({
        "status": "success",
        "message": "Fase 6: Kubernetes - Arquitectura Batch en progreso",
        "timestamp": datetime.datetime.now().isoformat(),
        "environment": os.getenv("ENV_ROLE", "Development-Local")
    })

@app.route('/healthz')
def health_check():
    # Este endpoint lo usará Kubernetes para los liveness/readiness probes más adelante
    return jsonify({"status": "healthy"}), 200

if __name__ == '__main__':
    # Escuchamos en el puerto 5000 en todas las interfaces de red
    app.run(host='0.0.0.0', port=5000)

In [ ]:
%%writefile requirements.txt
flask==3.0.2

In [ ]:
from flask import Flask, jsonify
import os
import datetime

app = Flask(__name__)

@app.route('/')
def home():
    return jsonify({
        "status": "success",
        "message": "Fase 6: Kubernetes - Arquitectura Batch en progreso",
        "timestamp": datetime.datetime.now().isoformat(),
        "environment": os.getenv("ENV_ROLE", "Development-Local")
    })

@app.route('/healthz')
def health_check():
    # Este endpoint lo usará Kubernetes para los liveness/readiness probes más adelante
    return jsonify({"status": "healthy"}), 200

if __name__ == '__main__':
    # Escuchamos en el puerto 5000 en todas las interfaces de red
    app.run(host='0.0.0.0', port=5000)


In [ ]:
!docker kill $(docker ps -q)

In [ ]:
!docker rm -f $(docker ps -a -q)

In [ ]:
!docker compose down --remove-orphans

In [ ]:
%%writefile Dockerfile
# Usamos una imagen base de Python ligera
FROM python:3.11-slim

# Establecemos el directorio de trabajo dentro del contenedor
WORKDIR /app

# Copiamos los requerimientos e instalamos dependencias
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copiamos nuestra aplicación
COPY app.py .

# Exponemos el puerto de la app (útil si la corremos como servicio)
EXPOSE 5000

# Comando por defecto
CMD ["python", "app.py"]

In [7]:
!docker build -t batch-app:v1 .

[+] Building 0.0s (0/1)                                                         
[+] Building 0.2s (2/2)                                                         
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 492B                                       0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
[+] Building 0.2s (10/10) FINISHED                                              
 => [internal] load build definition from Dockerfile                       0.0s
 => => transferring dockerfile: 492B                                       0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
 => [internal] load metadata for docker.io/library/python:3.11-slim        0.0s
 => [1/5] FROM docker.io/library/pyth

In [ ]:
%%writefile job-basico.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: batch-job-inicial
spec:
  template:
    spec:
      containers:
      - name: python-worker
        image: batch-app:v1
        imagePullPolicy: Never # Le dice a K8s que use la imagen local que acabamos de construir con Docker
        command: ["python", "app.py"] # En un Job real, aquí ejecutarías tu script de extracción o procesamiento de datos
        env:
        - name: ENV_ROLE
          value: "Batch-Job-Testing"
      restartPolicy: Never # Los Jobs de tipo Batch NO deben reiniciarse infinitamente si fallan
  backoffLimit: 2 # Número de reintentos antes de marcar el Job como fallido

In [1]:
!kubectl apply -f job-basico.yaml

job.batch/batch-job-inicial unchanged


In [ ]:
# Paso 6: Desplegar el Job en Kubernetes
!kubectl apply -f job-basico.yaml

job.batch/batch-job-inicial unchanged

# 1. Ver el estado del Job
!kubectl get jobs

NAME                COMPLETIONS   DURATION   AGE
batch-job-inicial   0/1           49m        49m

# 2. Ver el Pod que creó el Job
!kubectl get pods

NAME                      READY   STATUS    RESTARTS   AGE
batch-job-inicial-cktpw   1/1     Running   0          49m




In [3]:
# Paso 6: Desplegar el Job en Kubernetes
!kubectl apply -f job-basico.yaml

job.batch/batch-job-inicial unchanged


In [4]:
# 1. Ver el estado del Job
!kubectl get jobs


NAME                COMPLETIONS   DURATION   AGE
batch-job-inicial   0/1           49m        49m


In [5]:
# 2. Ver el Pod que creó el Job
!kubectl get pods

NAME                      READY   STATUS    RESTARTS   AGE
batch-job-inicial-cktpw   1/1     Running   0          49m


In [6]:
!kubectl logs -l job-name=batch-job-inicial 

 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.1.0.67:5000
Press CTRL+C to quit


In [8]:
%%bash
echo "🚀 --- DESPLEGANDO JOB EN KUBERNETES ---"

# 1. Borramos el job anterior si existía para que no choque por el nombre
kubectl delete job batch-job-inicial --ignore-not-found

# 2. Aplicamos el manifiesto del Job básico
kubectl apply -f job-basico.yaml

🚀 --- DESPLEGANDO JOB EN KUBERNETES ---
job.batch "batch-job-inicial" deleted
job.batch/batch-job-inicial created


In [9]:
%%bash
echo "📊 --- ESTADO DEL JOB Y LOS PODS ---"
kubectl get jobs
echo ""
kubectl get pods -l job-name=batch-job-inicial

📊 --- ESTADO DEL JOB Y LOS PODS ---
NAME                COMPLETIONS   DURATION   AGE
batch-job-inicial   0/1           78s        78s

NAME                      READY   STATUS    RESTARTS   AGE
batch-job-inicial-65srb   1/1     Running   0          79s


In [10]:
%%bash
echo "📋 --- LOGS DEL TRABAJADOR PYTHON (FLASK) ---"
kubectl logs -l job-name=batch-job-inicial --tail=50

📋 --- LOGS DEL TRABAJADOR PYTHON (FLASK) ---
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://10.1.0.68:5000
Press CTRL+C to quit


In [11]:
%%bash
# Iniciamos un port-forward en segundo plano del puerto 5000 de tu máquina al 5000 del Pod
kubectl port-forward job/batch-job-inicial 5000:5000 > /dev/null 2>&1 &
echo "🔗 ¡Túnel interactivo abierto con éxito!"
echo "👉 Abre tu navegador e ingresa a: http://127.0.0.1:5000"

🔗 ¡Túnel interactivo abierto con éxito!
👉 Abre tu navegador e ingresa a: http://127.0.0.1:5000


In [12]:
# recuerda de donde salio el nombre:

# 1. Ver el estado del Job
!kubectl get jobs


NAME                COMPLETIONS   DURATION   AGE
batch-job-inicial   0/1           9m7s       9m7s


In [13]:
%%writefile app.py
import os
import sys
import time
import datetime

def iniciar_procesamiento_batch():
    print(f"🚀 --- INICIANDO TAREA BATCH (FASE 6) ---")
    print(f"📅 Hora de inicio: {datetime.datetime.now().isoformat()}")
    print(f"🌐 Entorno detectado: {os.getenv('ENV_ROLE', 'Development-Local')}")
    
    # Simulamos el procesamiento de un lote de datos paso a paso
    total_registros = 100
    print(f"📦 Cargando {total_registros} registros para procesar...")
    time.sleep(2)
    
    print("⚡ Procesando bloque de datos 1/2 (Registros 1-50)...")
    time.sleep(3)
    
    print("⚡ Procesando bloque de datos 2/2 (Registros 51-100)...")
    time.sleep(3)
    
    print("✅ ¡Todos los registros fueron procesados y guardados con éxito!")
    print(f"🏁 Tarea finalizada a las: {datetime.datetime.now().isoformat()}")
    
    # Salida limpia y exitosa para avisar a Kubernetes que el Job terminó
    sys.exit(0)

if __name__ == '__main__':
    iniciar_procesamiento_batch()

Overwriting app.py


In [15]:
%%bash
echo "🐳 --- RECOMPILANDO IMAGEN BATCH ---"
docker build -t batch-app:v1 .

🐳 --- RECOMPILANDO IMAGEN BATCH ---


#1 [internal] load build definition from Dockerfile
#1 sha256:7f1f5b204acc8dfe3fb2c887993d3314adb0130213b344a2d247edcda0563943
#1 transferring dockerfile: 37B 0.0s done
#1 DONE 0.0s

#2 [internal] load .dockerignore
#2 sha256:6cc6e3ebafc8022fabe3877224e561047156104911f2b7d8597563df30d267f9
#2 transferring context: 2B done
#2 DONE 0.0s

#3 [internal] load metadata for docker.io/library/python:3.11-slim
#3 sha256:fec50dacbb21839eefab803f73dbb4d9623839a8f78459ebb826d917bc361c9b
#3 DONE 0.0s

#4 [1/5] FROM docker.io/library/python:3.11-slim
#4 sha256:393f53992145d4e66db06c6fc234cc485994eafdf13c53302514ff86dbc4b283
#4 DONE 0.0s

#6 [internal] load build context
#6 sha256:2db28210314463101dc5042b4182ac81f11d9942e41bc48bf3bf42b8aee69b0d
#6 transferring context: 63B done
#6 DONE 0.0s

#8 [4/5] RUN pip install --no-cache-dir -r requirements.txt
#8 sha256:531f6c0fb771a5d47959b97206dbf01a5a18925f1ee1a30e1b2095217bb77327
#8 CACHED

#5 [2/5] WORKDIR /app
#5 sha256:525f78dad77c7567087bd4575cca3d3bda

In [16]:
%%bash
echo "🚀 --- EJECUTANDO NUEVO JOB BATCH ---"

# 1. Limpiamos el Job Flask anterior
kubectl delete job batch-job-inicial --ignore-not-found

# 2. Volvemos a lanzar el Job usando el mismo manifiesto yaml
kubectl apply -f job-basico.yaml

🚀 --- EJECUTANDO NUEVO JOB BATCH ---
job.batch/batch-job-inicial created


In [17]:
%%bash
echo "📊 --- ESTADO ACTUAL DEL JOB ---"
kubectl get jobs
echo ""
echo "📋 --- LOGS DEL PROCESAMIENTO ---"
kubectl logs -l job-name=batch-job-inicial

📊 --- ESTADO ACTUAL DEL JOB ---
NAME                COMPLETIONS   DURATION   AGE
batch-job-inicial   1/1           14s        25s

📋 --- LOGS DEL PROCESAMIENTO ---
🚀 --- INICIANDO TAREA BATCH (FASE 6) ---
📅 Hora de inicio: 2026-07-04T20:56:53.402866
🌐 Entorno detectado: Batch-Job-Testing
📦 Cargando 100 registros para procesar...
⚡ Procesando bloque de datos 1/2 (Registros 1-50)...
⚡ Procesando bloque de datos 2/2 (Registros 51-100)...
✅ ¡Todos los registros fueron procesados y guardados con éxito!
🏁 Tarea finalizada a las: 2026-07-04T20:57:01.416609


In [18]:
%%writefile cronjob-basico.yaml
apiVersion: batch/v1
kind: CronJob
metadata:
  name: batch-cronjob-interactivo
spec:
  # Sintaxis Cron: (Minuto) (Hora) (Día del Mes) (Mes) (Día de la Semana)
  # '* * * * *' significa: Ejecutar CADA MINUTO de cada día.
  schedule: "* * * * *"
  
  # Si una tarea se retrasa, evita que se ejecuten múltiples instancias en paralelo
  concurrencyPolicy: Forbid
  
  # Cuántos Jobs completados y fallidos queremos guardar en el historial para ver logs
  successfulJobsHistoryLimit: 3
  failedJobsHistoryLimit: 2
  
  jobTemplate:
    spec:
      template:
        spec:
          containers:
          - name: python-worker
            image: batch-app:v1
            imagePullPolicy: Never  # Sigue usando tu imagen local compilada
            command: ["python", "app.py"]
            env:
            - name: ENV_ROLE
              value: "Batch-CronJob-Automation"
          restartPolicy: Never
      backoffLimit: 2

Writing cronjob-basico.yaml


In [20]:
%%bash
echo "⏰ --- ACTIVANDO EL CRONJOB EN KUBERNETES ---"

# Limpiamos si existía un cronjob anterior con el mismo nombre
kubectl delete cronjob batch-cronjob-interactivo --ignore-not-found

# Aplicamos el nuevo CronJob
kubectl apply -f cronjob-basico.yaml

⏰ --- ACTIVANDO EL CRONJOB EN KUBERNETES ---
cronjob.batch "batch-cronjob-interactivo" deleted
cronjob.batch/batch-cronjob-interactivo created


In [21]:
%%bash
echo "📊 --- LISTA DE CRONJOBS ACTIVOS ---"
kubectl get cronjobs
echo ""
echo "📦 --- JOBS GENERADOS AUTOMÁTICAMENTE ---"
kubectl get jobs

📊 --- LISTA DE CRONJOBS ACTIVOS ---
NAME                        SCHEDULE    SUSPEND   ACTIVE   LAST SCHEDULE   AGE
batch-cronjob-interactivo   * * * * *   False     1        6s              2m16s

📦 --- JOBS GENERADOS AUTOMÁTICAMENTE ---
NAME                                 COMPLETIONS   DURATION   AGE
batch-cronjob-interactivo-29719985   1/1           13s        2m6s
batch-cronjob-interactivo-29719986   1/1           13s        66s
batch-cronjob-interactivo-29719987   0/1           6s         6s
batch-job-inicial                    1/1           14s        10m


In [22]:
%%bash
echo "📋 --- LOGS DE LA INSTANCIA CRONJOB AUTOMÁTICA ---"
# Consultamos los logs usando el prefijo común de los pods del cronjob
kubectl logs -l job-name=batch-cronjob-interactivo-29719986

📋 --- LOGS DE LA INSTANCIA CRONJOB AUTOMÁTICA ---


No resources found in default namespace.


In [23]:
%%bash
echo "📋 --- CAPTURANDO LOS LOGS DE LA INSTANCIA MÁS RECIENTE ---"

# Buscamos el nombre del último pod creado por el cronjob y le sacamos los logs
ULTIMO_POD=$(kubectl get pods -l job-name --sort-by=.metadata.creationTimestamp -o jsonpath='{.items[-1].metadata.name}' 2>/dev/null)

if [ -z "$ULTIMO_POD" ]; then
    echo "⚠️ No se encontraron pods activos en este momento. Espera unos segundos a que el CronJob lance el siguiente."
else
    echo "🔍 Pod detectado: $ULTIMO_POD"
    echo "--------------------------------------------------------"
    kubectl logs $ULTIMO_POD
fi

📋 --- CAPTURANDO LOS LOGS DE LA INSTANCIA MÁS RECIENTE ---
🔍 Pod detectado: batch-cronjob-interactivo-29719998-qcpmp
--------------------------------------------------------
🚀 --- INICIANDO TAREA BATCH (FASE 6) ---
📅 Hora de inicio: 2026-07-04T21:18:02.577603
🌐 Entorno detectado: Batch-CronJob-Automation
📦 Cargando 100 registros para procesar...
⚡ Procesando bloque de datos 1/2 (Registros 1-50)...
⚡ Procesando bloque de datos 2/2 (Registros 51-100)...
✅ ¡Todos los registros fueron procesados y guardados con éxito!
🏁 Tarea finalizada a las: 2026-07-04T21:18:10.558766


In [24]:
%%bash
echo "🧹 --- LIMPIANDO ENTORNO BATCH AUTOMATIZADO ---"
kubectl delete cronjob batch-cronjob-interactivo
kubectl delete job batch-job-inicial --ignore-not-found

🧹 --- LIMPIANDO ENTORNO BATCH AUTOMATIZADO ---
cronjob.batch "batch-cronjob-interactivo" deleted
job.batch "batch-job-inicial" deleted


In [25]:
# kubectl
import os
print(os.getcwd())

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container


In [26]:
%%writefile app.py
import time
import os

def run_batch():
    # El eje inferior izquierdo: Variables de entorno inyectadas por Kubernetes
    batch_name = os.getenv("BATCH_NAME", "Lote-Default")
    total_items = int(os.getenv("TOTAL_ITEMS", "5"))
    
    print(f"=== INICIANDO: {batch_name} ===")
    print("Estado: Punto de equilibrio alcanzado. Procesando...")
    
    for i in range(1, total_items + 1):
        print(f" -> Procesando elemento {i}/{total_items}...")
        time.sleep(1)
        
    print(f"=== TRABAJO COMPLETADO CON ÉXITO ===")

if __name__ == "__main__":
    run_batch()

Writing app.py


In [27]:
%%writefile pod.yaml
apiVersion: v1
kind: Pod
metadata:
  name: python-batch-pod
  labels:
    component: architect-python-batch
    layer: exercise-1
spec:
  restartPolicy: Never
  containers:
  - name: python-worker
    image: python:3.10-slim
    command: ["python", "-c", "import os; print('Simulación de entorno lista')"]
    # En el siguiente paso acoplaremos el script completo, 
    # por ahora probemos que la estructura base levanta correctamente.

Writing pod.yaml
